# No-anchor 6/20 synthetic integration smoke

快速验证 F=D、E/L、registry hash lookup、exact/learned TB、梯度隔离、same-N retry、no-anchor checkpoint 确定性恢复和旧 schema 拒绝。仅使用 synthetic Reward，不证明真实数据可运行。

In [ ]:
import sys, tempfile, torch
from dataclasses import replace
from pathlib import Path
root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'factor_gfn').is_dir())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from factor_gfn.gfn import GFNTrainer, ModelConfig, NoAnchorComplexityConfig, NoAnchorGFNConfig, SearchSpaceConfig, SyntheticRewardProvider, TrainingConfig
from factor_gfn.gfn.diagnostic_support import build_or_resume_n1_n2_registry, configure_registry_once, run_training_with_progress
RUN_SYNTHETIC_SMOKE = True
assert RUN_SYNTHETIC_SMOKE
print('[setup] synthetic CPU smoke enabled', flush=True)

In [ ]:
work = Path(tempfile.mkdtemp(prefix='no_anchor_smoke_'))
provider = SyntheticRewardProvider()
config = NoAnchorGFNConfig(search_space=SearchSpaceConfig(max_depth=6, max_nodes=20), model=ModelConfig(d_model=16, num_heads=4, num_layers=1, dim_feedforward=32, dropout=0.0, token_policy_mode='grammar_hierarchical'), training=TrainingConfig(batch_size=8, max_steps=5, learning_rate=1e-3, log_z_learning_rate=1e-2, seed=20260813))
registry = build_or_resume_n1_n2_registry(work / 'registry.sqlite3', provider, reward_floor=config.reward.reward_floor, progress_every=25)
trainer = GFNTrainer(config, provider, device='cpu')
configure_registry_once(trainer, registry)
assert trainer.resolved_discovery_node_counts == trainer.resolved_feasible_node_counts
assert trainer.resolved_exhaustive_node_counts == (1, 2)
assert trainer.resolved_learned_node_counts == tuple(range(3, 21))
for slot in range(64 * len(trainer.resolved_learned_node_counts)):
    n = trainer.calibration.next_node_count()
    trainer.calibration.record_slot(n, sampled_attempts=1, implied_log_z=1.0 + n / 100.0)
    if (slot + 1) % 20 == 0: print(f'[calibration-synthetic] {slot + 1}/{64 * 18}', flush=True)
trainer._finalize_calibration()
assert trainer.calibration.status == 'complete'
print('[calibration-synthetic] complete; synthetic constant windows intentionally accelerate this smoke', flush=True)

In [ ]:
exact_before = trainer.tb_loss.exact_tb_log_z_by_node_count.clone()
rows, candidates = run_training_with_progress(trainer, logical_batches=2)
assert torch.equal(exact_before, trainer.tb_loss.exact_tb_log_z_by_node_count)
checkpoint = work / 'no_anchor.pt'
trainer.save_checkpoint(checkpoint)
payload = torch.load(checkpoint, map_location='cpu', weights_only=False)
assert payload['schema'] == 'factor_gfn.checkpoint.no_anchor.v1'
assert not ({'anchor_state', 'anchor_optimizer_step', 'total_policy_optimizer_step'} & set(payload))
expected = trainer.train_step(); expected_diag = trainer.last_discovery_trajectory_diagnostics
expected_model = {k: v.detach().clone() for k, v in trainer.model.state_dict().items()}; expected_log_z = trainer.tb_loss.log_z_by_node_count.detach().clone(); expected_scheduler = trainer.complexity_scheduler.state_dict()
resumed = GFNTrainer(config, provider, device='cpu'); configure_registry_once(resumed, registry); resumed.load_checkpoint(checkpoint)
actual = resumed.train_step()
assert actual == expected
assert resumed.last_discovery_trajectory_diagnostics == expected_diag
assert resumed.complexity_scheduler.state_dict() == expected_scheduler
assert torch.equal(resumed.tb_loss.log_z_by_node_count, expected_log_z)
assert all(torch.equal(resumed.model.state_dict()[k], v) for k, v in expected_model.items())
legacy = dict(payload); legacy['schema'] = 'factor_gfn.checkpoint.v5'; torch.save(legacy, work / 'legacy.pt')
try:
    resumed.load_checkpoint(work / 'legacy.pt')
    raise AssertionError('legacy schema was not rejected')
except ValueError as error:
    assert 'rejects legacy' in str(error)
print('[checkpoint] deterministic resume and strict legacy rejection OK', flush=True)

In [ ]:
from factor_gfn.gfn import RewardAssignment
class RejectFirst(SyntheticRewardProvider):
    def __init__(self): super().__init__(); self.calls = 0
    def evaluate(self, expression):
        self.calls += 1
        if self.calls == 1: return RewardAssignment(valid=False, rejection_reason='synthetic_first_failure')
        return super().evaluate(expression)
retry_provider = RejectFirst()
retry_config = replace(config, complexity=NoAnchorComplexityConfig(exact_normalizer_node_counts=(), exact_node_retry_budget=1), training=replace(config.training, batch_size=1, max_steps=1, seed=17))
retry_trainer = GFNTrainer(retry_config, retry_provider, device='cpu')
for _ in range(64 * len(retry_trainer.resolved_learned_node_counts)):
    n = retry_trainer.calibration.next_node_count(); retry_trainer.calibration.record_slot(n, sampled_attempts=1, implied_log_z=1.0)
retry_trainer._finalize_calibration()
retry_stats = retry_trainer.train_step()
assigned_n = next(n for n, count in retry_stats.requested_count_by_N.items() if count)
assert retry_stats.sampled_attempt_count_by_N[assigned_n] == 2
assert retry_stats.valid_count_by_N[assigned_n] == 1
assert not retry_stats.skipped_update
registry.close()
print('NO_ANCHOR_SYNTHETIC_SMOKE_OK', flush=True)
print('temporary result:', work, flush=True)